In [1]:
print("hi how r u ")

hi how r u 


# Biggest Heading (H1)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   
from wordcloud import WordCloud, STOPWORDS
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/macsolutions/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/macsolutions/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [9]:
df = pd.read_csv(r"/Users/macsolutions/Desktop/YOUTUBE-MLOOPS-SERIES/YL-MLOOPS-PIPELINE/experiments/spam.csv")

In [10]:
df.head(3)

,UDI,Product_ID,Type,Air_temperature_K,Process_temperature_K,Rotational_speed_rpm,Torque_Nm,Tool_wear_min,Machine_Failure
0,1,M_25795,M,299.0,310.4,1575,29.7,211,0
1,2,M_10860,H,300.3,309.9,1579,19.5,33,0
2,3,M_86820,M,300.4,310.3,1414,55.6,59,0


In [12]:
df.isnull().sum()


UDI                      0
Product_ID               0
Type                     0
Air_temperature_K        0
Process_temperature_K    0
Rotational_speed_rpm     0
Torque_Nm                0
Tool_wear_min            0
Machine_Failure          0
dtype: int64

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin

# Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# NLP Tools for Stemming
import nltk
from nltk.stem import PorterStemmer
nltk.download('punkt', quiet=True)

# 1. CUSTOM FEATURE ENGINEERING
def create_features(df):
    df = df.copy()
    # Create a 'Temperature Difference' feature
    df['Temp_Diff'] = df['Process_temperature_K'] - df['Air_temperature_K']
    # Create a 'Power' feature (Torque * Speed)
    df['Power_Index'] = df['Torque_Nm'] * df['Rotational_speed_rpm']
    return df

# 2. CUSTOM STEMMING TOKENIZER (For Text Data)
class StemmedTfidf(TfidfVectorizer):
    def build_analyzer(self):
        analyzer = super(StemmedTfidf, self).build_analyzer()
        stemmer = PorterStemmer()
        return lambda doc: (stemmer.stem(w) for w in analyzer(doc))

# --- DATA PREPARATION ---
df = pd.read_csv('spam.csv')

# Adding a dummy text column just to demonstrate the Tokenizer/Stemming pipeline
# In a real scenario, this would be your "Error Messages" or "Logs" column
df['Error_Logs'] = "machine operating normally" 

X = df.drop(columns=['UDI', 'Product_ID', 'Machine_Failure'])
y = df['Machine_Failure']

# --- DEFINING THE PIPELINE STEPS ---

# Identify columns
numeric_cols = ['Air_temperature_K', 'Process_temperature_K', 'Rotational_speed_rpm', 'Torque_Nm', 'Tool_wear_min', 'Temp_Diff', 'Power_Index']
categorical_cols = ['Type']
text_col = 'Error_Logs'

# Preprocessing for Numeric, Categorical, and Text data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('txt', StemmedTfidf(stop_words='english'), text_col) # Stemming + Tokenization
    ])

# Create the master pipeline structure
def build_master_pipeline(classifier):
    return Pipeline(steps=[
        ('feature_eng', FunctionTransformer(create_features)), # Feature Engineering
        ('preprocessor', preprocessor),                       # Transforming & Tokenizing
        ('classifier', classifier)                             # The Model
    ])

# --- MODEL SELECTION ---
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'SVM': SVC(probability=True),
    'KNN': KNeighborsClassifier()
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- EXECUTION & RESULTS ---
final_results = []

print(f"{'Model Name':<20} | {'Train Acc':<10} | {'Test Acc':<10}")
print("-" * 45)

for name, model in models.items():
    pipe = build_master_pipeline(model)
    pipe.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    test_acc = accuracy_score(y_test, pipe.predict(X_test))
    
    final_results.append({'Model': name, 'Train': train_acc, 'Test': test_acc})
    print(f"{name:<20} | {train_acc:<10.4f} | {test_acc:<10.4f}")

# --- OPTIONAL: HYPER-TUNING THE BEST MODEL ---
# Example for Gradient Boosting
param_grid = {'classifier__n_estimators': [50, 100], 'classifier__learning_rate': [0.01, 0.1]}
grid = GridSearchCV(build_master_pipeline(GradientBoostingClassifier()), param_grid, cv=3)
grid.fit(X_train, y_train)
print(f"\nBest Tuned Parameters: {grid.best_params_}")

Model Name           | Train Acc  | Test Acc  
---------------------------------------------
Random Forest        | 1.0000     | 0.9925    
Gradient Boosting    | 0.9988     | 0.9925    
Logistic Regression  | 0.9812     | 0.9850    
AdaBoost             | 0.9881     | 0.9925    
SVM                  | 0.9756     | 0.9600    
KNN                  | 0.9325     | 0.9000    

Best Tuned Parameters: {'classifier__learning_rate': 0.01, 'classifier__n_estimators': 100}
